# Bước B — Lấy một câu hỏi và gọi một model

**Đích hôm nay:** `questions.jsonl → một câu hỏi → Qwen → predictions.jsonl`.

Chạy từng ô từ trên xuống, xem đầu ra rồi mới sang ô tiếp. Cần gói ZIP **đã cập nhật có `unit02_model.py`** từ tin nhắn bước B. Chưa làm RAG, SFT/DPO hoặc chấm điểm.

Model mẫu: `Qwen/Qwen2.5-1.5B-Instruct`. Chưa chọn model cuối cho khóa luận.

Trước khi sinh: vào **Runtime → Change runtime type → GPU**. Notebook kiểm tra GPU ở ô cài thư viện. Colab có thể không cấp được GPU; khi đó dừng và báo lại.

Phần code/định dạng notebook đã được kiểm tra; tác giả chưa chạy model thật trong môi trường tạo file. Đầu ra của notebook đang để trống, không phải kết quả benchmark có sẵn.


## Ô 1 — Đưa gói code và câu hỏi lên Colab

Chọn `Gout_Build_tung_unit_v01.zip` bản mới. Chương trình giải nén vào thư mục mới và không ghi đè dữ liệu của lần chạy trước.


In [ ]:
from google.colab import files
from pathlib import Path
import io, json, tempfile, zipfile, subprocess, sys

uploaded = files.upload()
archives = [(name, data) for name, data in uploaded.items() if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Chọn đúng một file ZIP của bước B.')
WORK = Path(tempfile.mkdtemp(prefix='gout_step_b_'))
with zipfile.ZipFile(io.BytesIO(archives[0][1])) as archive:
    for item in archive.infolist():
        target = (WORK / item.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError('Đường dẫn trong ZIP không hợp lệ.')
        if (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('ZIP không được chứa symlink.')
    archive.extractall(WORK)
PROJECT = WORK / 'gout_units_v01'
if not (PROJECT / 'unit02_model.py').is_file():
    raise ValueError('Đây là ZIP cũ. Tải lại ZIP có unit02_model.py từ tin nhắn bước B.')
print('Đã nạp project:', PROJECT)


## Ô 2 — Chọn đúng một câu hỏi từ đầu ra A

Gói có sẵn bốn lượt của nhóm đầu tiên, giống đầu ra bước A. Lần này chỉ chọn câu **đơn lượt** `GOUT_ST_001__T1`.


In [ ]:
QUESTIONS = PROJECT / 'examples/unit01/questions.jsonl'
SAMPLE_ID = 'GOUT_ST_001__T1'
rows = [json.loads(line) for line in QUESTIONS.read_text(encoding='utf-8').splitlines() if line.strip()]
selected = [row for row in rows if row['sample_id'] == SAMPLE_ID]
if len(selected) != 1:
    raise ValueError('Không tìm thấy duy nhất câu hỏi đã chọn.')
print('Mã:', selected[0]['sample_id'])
print('Câu hỏi:', selected[0]['user'])


## Ô 3 — Xem messages sẽ đưa cho model

Chưa tải hay gọi model. `messages` gồm một system prompt và một câu hỏi. Các ID/category chỉ dùng ghi log, không đưa vào prompt.


In [ ]:
preview_out = Path(tempfile.mkdtemp(prefix='gout_preview_')) / 'run'
subprocess.run([
    sys.executable, '-u', str(PROJECT / 'unit02_model.py'),
    '--questions', str(QUESTIONS), '--sample-id', SAMPLE_ID,
    '--preview', '--out', str(preview_out)
], check=True)
request = json.loads((preview_out / 'request.json').read_text(encoding='utf-8'))
for message in request['messages']:
    print('\nROLE:', message['role'])
    print(message['content'])


## Ô 4 — Kiểm tra GPU và cài thư viện

Chỉ chạy một model. Ô này dùng PyTorch đã có trên Colab; không tự đổi phiên bản CUDA. Nếu vừa đổi runtime, hãy chạy lại các ô trước vì trạng thái Python có thể đã mất.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → GPU rồi thử lại.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT / 'requirements-unit02.txt')
], check=True)
print('Đã cài thư viện cho bước B.')


## Ô 5 — Nạp model và sinh một câu trả lời thật

Lần đầu sẽ tải trọng số từ Hugging Face. Script dùng chat template của model, chỉ lấy token mới của phần trả lời. Mỗi lần chạy ô này tạo một thư mục kết quả mới.

Đây là lần gọi model duy nhất của bài thử. Nếu gặp lỗi, đọc thông báo và vẫn chạy ô tải kết quả để giữ log.


In [ ]:
RUN = Path(tempfile.mkdtemp(prefix='gout_result_')) / 'run'
print('Bắt đầu tải/nạp model; lần đầu cần chờ tải trọng số.', flush=True)
process = subprocess.run([
    sys.executable, '-u', str(PROJECT / 'unit02_model.py'),
    '--questions', str(QUESTIONS), '--sample-id', SAMPLE_ID,
    '--out', str(RUN)
], check=False)
print('Mã thoát:', process.returncode)
print('Thư mục kết quả:', RUN)
if process.returncode != 0:
    print('Lần chạy chưa thành công. Xem lỗi phía trên và tải log ở ô cuối.')


## Ô 6 — Đọc câu trả lời và xác định bước B đã chạy được chưa

Chỉ chuyển sang C khi có `status=ok`, answer không rỗng và model revision đã được ghi. Chưa chấm đúng/sai y khoa ở đây. Nếu `finish_reason=length`, output có thể bị cắt do hết token.


In [ ]:
prediction_path = RUN / 'predictions.jsonl'
if prediction_path.exists():
    result = json.loads(prediction_path.read_text(encoding='utf-8'))
    print('STATUS:', result['status'])
    print('QUESTION:', result['question'])
    print('ANSWER:\n', result['answer'])
    print('MODEL META:', result.get('model_meta'))
    print('GENERATION MS:', result.get('generation_ms'))
    if result['status'] == 'error':
        print('ERROR:', result.get('error'))
else:
    print('Chưa có prediction; xem thông báo tiến trình và manifest nếu có.')
manifest_path = RUN / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('BACKEND:', manifest.get('backend_info'))


## Ô 7 — Tải kết quả về máy

Tải gói này trước khi tắt Colab. Có thể gửi lại `predictions.jsonl` hoặc ZIP kết quả để mình cùng đọc câu trả lời và chuyển sang phần lịch sử hội thoại.


In [ ]:
result_zip = RUN.parent / 'Gout_Buoc_B_ket_qua.zip'
if not RUN.is_dir():
    raise RuntimeError('Chưa tạo được thư mục run. Hãy xem lỗi ở ô 5.')
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(RUN))
files.download(str(result_zip))


Tài liệu: [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Transformers chat templates](https://huggingface.co/docs/transformers/v4.57.1/en/chat_templating), [Colab FAQ](https://research.google.com/colaboratory/faq.html).

Sau khi chạy được, mở `README_BUOC_B.md` để hiểu từng hàm tương ứng B01–B08.
